# Packages import 

In [ ]:
import os
import re
import yaml 
import requests
import pandas as pd
from requests.auth import HTTPBasicAuth
from bs4 import BeautifulSoup

ModuleNotFoundError: No module named 'yaml'

# Apollo scraper

In [ ]:
group_id = input("Enter group ID: ")
#252671
url = f'https://planzajec.uek.krakow.pl/index.php?typ=G&id={group_id}&okres=2'

In [ ]:
with open("config.yaml", "r", encoding="UTF-8") as yf:
    config =yaml.safe_load(yf)
username = config["credentials"]['username']
password = config["credentials"]['password']

auth = HTTPBasicAuth(username, password)

In [ ]:
response = requests.get(url, auth=auth)
response.encoding = "UTF-8"
print(response.status_code)

200


In [ ]:
page_dom = BeautifulSoup(response.text, 'html.parser')

In [ ]:
group_name = page_dom.select_one("div.grupa").get_text()
print(group_name)

ZICSS1-1212


In [ ]:
classes_tag = page_dom.select_one("table")
with open("temp.html", "w", encoding="UTF-8") as hf:
    hf.write(str(classes_tag))
classes = pd.read_html("temp.html", encoding= "UTF-8")[0]
os.remove("temp.html")

In [ ]:
classes = classes.loc[classes['Typ'].isin(['ćwiczenia', 'wykład', 'egzamin'])]

In [ ]:
classes[['Day', 'Start time', 'Hyphen', 'End time', 'Duration']] = classes['Dzień, godzina'].str.split(' ',expand=True)

In [ ]:
classes["Duration"] = classes["Duration"].map(lambda x: x.split("(")[1].split("g"[0]))

In [ ]:
classes = classes.drop(["Hyphen"], axis=1)

In [ ]:
classes["Sala"] = classes['Sala'].str.replace(
    r"(lab\.).*",
    r"\1",
    regex=True
)

In [ ]:
if not os.path.exists("./schedules"):
    os.mkdir("./schedules")

In [ ]:
classes.to_csv(f"schedules/{group_name}.csv", encoding="UTF-8")